# Pipeline complet IBKR — options ASML de bout en bout

Reconstruction A→Z d'une **vraie** chaîne d'options (ASML, EUREX, delayed gratuit via IB
Gateway), **100 % offline** : on rejoue un échantillon **committé** de données IBKR réelles —
aucune connexion broker, aucun réseau, aucun Gateway. Le notebook ne fait qu'**importer et
appeler** le moteur testé : `storage.events_from_json` rejoue l'échantillon dans une couche
raw éphémère, puis `orchestration.reconstruction.reconstruct_day` exécute **le seul** chemin de
calcul de la prod (snapshots → QC → forward → inversion IV → surface SVI), et le notebook trace
les sorties typées (`ActorOutputs`). **Aucune formule** n'est écrite ici.

> **Périmètre de l'échantillon** : **4 échéances** (juillet–août 2026, ~20 strikes ATM chacune)
> → smile par maturité, fit SVI **et surface 3D** (≥ 2 maturités). Contrairement à Saxo, **IBKR
> ne fournit aucun Greek ni IV** sur le flux delayed : le moteur reconstruit tout (forward, IV)
> depuis les seules quotes.

> **Live (credential-gated)** : une cellule en fin de notebook montre le chemin flux-réel via
> `orchestration.run_provider_flow` ; elle est **désactivée par défaut** (`RUN_LIVE = False`) et
> nécessite un IB Gateway. Le chemin par défaut ci-dessous tourne **sans aucun identifiant**.

In [ ]:
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

C = {
    "blue": "#2563EB",  # primary series / calls
    "teal": "#0D9488",  # puts / complementary
    "violet": "#7C3AED",  # gamma / vega
    "amber": "#D97706",  # warning / stress
    "red": "#DC2626",  # rejected / violation
    "green": "#16A34A",  # usable / pass
    "indigo": "#4F46E5",
    "slate900": "#0F172A",
    "slate600": "#475569",
    "slate400": "#94A3B8",
    "slate100": "#F1F5F9",
    "white": "#FFFFFF",
}
DISCRETE = [C["blue"], C["teal"], C["violet"], C["amber"], C["indigo"], "#0EA5E9", "#F59E0B"]
FONT = "Inter, IBM Plex Sans, -apple-system, sans-serif"

SURFACE_COLORSCALE = [
    [0.00, "#1E3A5F"],
    [0.25, "#2563EB"],
    [0.50, "#0D9488"],
    [0.75, "#D97706"],
    [1.00, "#DC2626"],
]
SURFACE_CAMERA = dict(eye=dict(x=1.6, y=-1.6, z=0.9), up=dict(x=0, y=0, z=1))
SURFACE_ASPECT = dict(x=1.5, y=1.2, z=0.7)

_axis = dict(
    showgrid=True,
    gridcolor=C["slate400"],
    gridwidth=0.5,
    zeroline=False,
    linecolor=C["slate400"],
    linewidth=1,
    ticks="outside",
    tickcolor=C["slate400"],
    tickfont=dict(size=11),
    title_font=dict(size=12, color=C["slate600"]),
)
_tmpl = go.layout.Template()
_tmpl.layout = go.Layout(
    font=dict(family=FONT, size=12, color=C["slate600"]),
    title_font=dict(family=FONT, size=15, color=C["slate900"]),
    paper_bgcolor=C["white"],
    plot_bgcolor=C["white"],
    colorway=DISCRETE,
    xaxis=_axis,
    yaxis=_axis,
    legend=dict(
        bgcolor="rgba(255,255,255,0.85)",
        bordercolor=C["slate400"],
        borderwidth=1,
        font=dict(size=11),
        tracegroupgap=4,
    ),
    margin=dict(l=64, r=24, t=56, b=56),
    hoverlabel=dict(
        bgcolor=C["slate900"],
        font=dict(color=C["white"], size=12, family=FONT),
        bordercolor=C["slate900"],
    ),
    hovermode="closest",
)
pio.templates["algotrading"] = _tmpl
pio.templates.default = "plotly_white+algotrading"
print("Chart config loaded — template: algotrading (Plotly V2)")

## 0 · Rejeu de l'échantillon committé
Un seul appel : sample EAV → couche raw éphémère → `reconstruct_day` (le moteur acteur). Tout le reste lit `outputs`.

In [ ]:
import logging
import tempfile
from collections import defaultdict
from datetime import date
from pathlib import Path

import numpy as np
from algotrading.core.config import (
    PlatformConfig,
    QcThresholdConfig,
    ScenarioConfig,
    SolverConfig,
    UniverseConfig,
)
from algotrading.infra.contracts import InstrumentMaster, RawMarketEvent, content_event_id
from algotrading.infra.contracts.instrument_key import InstrumentKey
from algotrading.infra.orchestration.reconstruction import reconstruct_day
from algotrading.infra.storage import ParquetStore, events_from_json
from algotrading.infra.surfaces import SviParams, summarize_surface_parameters
from algotrading.infra.universe import parse_instrument_key
from algotrading.infra.universe.contracts import Underlying

# The pipeline emits structured QC logs to stderr; raise the floor so only the
# notebook's own print() lines show. Not a change to the engine.
logging.getLogger("algotrading").setLevel(logging.ERROR)

# Quote fields the actor consumes to rebuild snapshots (bid/ask/last -> reference
# spot). Any broker-supplied Greek/IV column is intentionally not replayed into the
# compute: the engine reconstructs forward, IV and Greeks from the quotes alone.
_QUOTE_FIELDS = ("bid", "ask", "last")


def _repo_root() -> Path:
    for p in (Path.cwd(), *Path.cwd().parents):
        if (p / "pyproject.toml").exists() and (p / "packages").is_dir():
            return p
    raise FileNotFoundError("repo root not found (pyproject.toml + packages/)")


def _instrument_key_of(colon_key: str, broker_id: str) -> InstrumentKey:
    """Relabel a sample's canonical colon key into the contracts pipe-form key.

    The committed samples carry the universe colon-form key (``OPT:ASML:...``); the
    actor matches snapshots against the contracts ``InstrumentKey`` and reads
    strike/expiry/right off the master. Pure relabelling of one identity into the
    other -- no analytics, no market data invented.
    """
    domain = parse_instrument_key(colon_key)
    if isinstance(domain, Underlying):
        return InstrumentKey(
            underlying_symbol=domain.symbol,
            security_type=domain.security_type,
            exchange=domain.exchange,
            currency=domain.currency,
            multiplier=1.0,
            broker_contract_id=broker_id,
        )
    return InstrumentKey(
        underlying_symbol=domain.symbol,
        security_type=domain.security_type,
        exchange=domain.exchange,
        currency=domain.currency,
        multiplier=float(domain.multiplier),
        broker_contract_id=broker_id,
        expiry=domain.expiry,
        strike=float(domain.strike),
        option_right=domain.right.value,
    )


def demo_config(underlying: str, exchange: str) -> PlatformConfig:
    """A permissive demo config for replaying a single committed slice off-session.

    The quote-age ceiling is wide because a captured sample is, by construction,
    older than now; the rest mirror the engine defaults. Business parameters live in
    the typed config, never as literals scattered through the analytics.
    """
    return PlatformConfig(
        universe=UniverseConfig(version="demo-u", underlyings=(underlying,), exchange=exchange),
        qc_threshold=QcThresholdConfig(
            version="demo-qc", max_spread_pct=0.5, max_quote_age_seconds=86_400.0, min_chain_count=1
        ),
        solver=SolverConfig(version="demo-iv", iv_tolerance=1e-12, max_iterations=200),
        scenario=ScenarioConfig(
            version="demo-scn", spot_shocks=(-0.05, 0.05), vol_shocks=(0.05, -0.05)
        ),
    )


def replay_sample(sample_path, *, underlying, exchange, config_hash="demo-notebook"):
    """Replay a committed real sample through the one actor pipeline, fully offline.

    Reads the EAV sample (``events_from_json``), relabels each instrument key to its
    contracts form, builds the day's masters, seeds an ephemeral ``ParquetStore`` raw
    layer, and runs ``reconstruct_day`` -- the *same* compute path production runs live.
    No broker, no network, no token. Returns ``(outputs, as_of, masters, config)``.
    """
    config = demo_config(underlying, exchange)
    storage_events = events_from_json(Path(sample_path).read_text(encoding="utf-8"))
    as_of = max(e.receipt_ts for e in storage_events)
    trade_date = as_of.date()

    broker_id_by_key: dict[str, str] = {}
    for e in storage_events:
        broker_id_by_key.setdefault(e.instrument_key, e.contract_id_broker or e.instrument_key)
    instrument_by_key = {
        colon: _instrument_key_of(colon, broker_id) for colon, broker_id in broker_id_by_key.items()
    }
    canonical = {colon: ik.canonical() for colon, ik in instrument_by_key.items()}

    contract_events: list[RawMarketEvent] = []
    sequence: dict[tuple[str, str], int] = {}
    for e in storage_events:
        if e.field_value is None or e.field_name not in _QUOTE_FIELDS:
            continue
        key = canonical[e.instrument_key]
        seq = sequence.get((key, e.field_name), 0)
        sequence[(key, e.field_name)] = seq + 1
        canonical_ts = e.exchange_ts or e.receipt_ts
        contract_events.append(
            RawMarketEvent(
                session_id=e.collector_session_id,
                event_id=content_event_id(key, e.field_name, seq),
                instrument_key=key,
                exchange_ts=canonical_ts,
                receipt_ts=e.receipt_ts,
                canonical_ts=canonical_ts,
                field_name=e.field_name,
                value=float(e.field_value),
                trade_date=trade_date,
                underlying=e.underlying,
            )
        )

    instruments = list(instrument_by_key.values())
    masters = [
        InstrumentMaster(
            instrument_key=ik.canonical(),
            as_of_date=trade_date,
            instrument=ik,
            raw_broker_payload="{}",
        )
        for ik in instruments
    ]

    with tempfile.TemporaryDirectory() as tmp:
        store = ParquetStore(Path(tmp) / "store")
        store.write("raw_market_events", contract_events)
        day = reconstruct_day(
            store,
            trade_date,
            [],
            instruments=instruments,
            masters=masters,
            config=config,
            config_hash=config_hash,
            as_of=as_of,
            calc_ts=as_of,
            persist=False,
        )
    return day.outputs, as_of, masters, config


def maturity_years(expiry: date, as_of_date: date) -> float:
    """ACT/365 year fraction -- the one day-count the actor solves maturity under."""
    return (expiry - as_of_date).days / 365.0


REPO = _repo_root()
SYMBOL = "ASML"
EXCHANGE = "EUREX"
CURRENCY = "EUR"
SAMPLE = REPO / "packages/infra-ibkr/samples/asml_real_2026-06-05.json"

# One call: committed sample -> the one actor pipeline -> typed ActorOutputs. No broker.
outputs, AS_OF, masters, CONFIG = replay_sample(SAMPLE, underlying=SYMBOL, exchange=EXCHANGE)
MASTER_BY_KEY = {m.instrument_key: m.instrument for m in masters}

print(f"Repo        : {REPO}")
print(f"Sample      : {SAMPLE.relative_to(REPO)}")
print(f"Underlying  : {SYMBOL} (EUREX, {CURRENCY})")
print(f"As-of       : {AS_OF}")
print(
    "Outputs     : "
    f"{len(outputs.snapshots)} snapshots, {len(outputs.forwards)} forwards, "
    f"{len(outputs.iv_points)} IV points, {len(outputs.surface_parameters)} SVI slices"
)

## 1 · Snapshot & contrôle qualité (QC)
Le moteur agrège le dernier état par instrument/champ et filtre **avant** forward/IV/surface.
`outputs.snapshots` est l'ensemble des snapshots persistés (référence spot, spread, complétude).

In [ ]:
snapshots = outputs.snapshots
spot_snap = next((s for s in snapshots if not MASTER_BY_KEY[s.instrument_key].is_option()), None)
ref_spot = float(spot_snap.reference_spot) if spot_snap else float("nan")
print(f"Snapshots construits : {len(snapshots)}")
print(
    f"Spot reference       : {ref_spot:,.2f} {CURRENCY} "
    f"({spot_snap.reference_type if spot_snap else 'n/a'})"
)

# Spread % par strike, depuis les snapshots d'options (le moteur a déjà choisi le spot ref).
strikes, spreads = [], []
for s in snapshots:
    instrument = MASTER_BY_KEY[s.instrument_key]
    if not instrument.is_option() or s.bid is None or s.ask is None:
        continue
    mid = (float(s.bid) + float(s.ask)) / 2.0
    if mid > 0:
        strikes.append(instrument.strike)
        spreads.append((float(s.ask) - float(s.bid)) / mid * 100.0)
strikes_arr, spreads_arr = np.array(strikes), np.array(spreads)
print(f"Options avec quote deux-faces : {len(strikes_arr)}")

fig = make_subplots(
    cols=2,
    column_widths=[0.78, 0.22],
    shared_yaxes=True,
    subplot_titles=("Spread % par strike", "Distribution"),
)
fig.add_trace(
    go.Scatter(
        x=strikes_arr,
        y=spreads_arr,
        mode="markers",
        marker=dict(color=C["blue"], size=6, opacity=0.75, line=dict(color=C["white"], width=0.5)),
        name="Option",
        hovertemplate="<b>Strike %{x:.0f}</b><br>Spread: %{y:.2f}%<extra></extra>",
    ),
    row=1,
    col=1,
)
fig.add_trace(
    go.Histogram(y=spreads_arr, nbinsy=30, marker_color=C["blue"], opacity=0.7, showlegend=False),
    row=1,
    col=2,
)
if not np.isnan(ref_spot):
    fig.add_vline(x=ref_spot, line_dash="dash", line_color=C["slate400"], row=1, col=1)
fig.update_layout(
    title=f"{SYMBOL} — Spread bid-ask % par strike",
    yaxis_title="Spread (%)",
    xaxis_title=f"Strike ({CURRENCY})",
    height=380,
    barmode="overlay",
)
fig.show()

## 2 · Courbe forward (parité put-call)
Le forward implicite par échéance, déduit de la parité call/put sur les quotes usables.
`outputs.forwards` porte un `ForwardCurvePoint` par maturité retenue.

In [ ]:
forwards = sorted(outputs.forwards, key=lambda f: f.maturity_years)
print(f"Maturites avec forward : {len(forwards)}")
for f in forwards:
    dte = int(f.maturity_years * 365)
    print(f"  {f.expiry_date}  ({dte:3d}j)  F={float(f.forward):,.2f} {CURRENCY}")

if forwards:
    dtes = [int(f.maturity_years * 365) for f in forwards]
    basis = [float(f.forward) - ref_spot for f in forwards]
    labels = [str(f.expiry_date) for f in forwards]
    fig = go.Figure()
    fig.add_trace(
        go.Scatter(
            x=dtes,
            y=basis,
            mode="lines+markers",
            line=dict(color=C["blue"], width=2),
            marker=dict(size=8),
            customdata=labels,
            hovertemplate="<b>%{customdata}</b><br>DTE: %{x}j<br>Basis: %{y:,.2f}<extra></extra>",
            name="F - Spot",
        )
    )
    fig.add_hline(y=0, line_dash="dash", line_color=C["slate400"])
    fig.update_layout(
        title=f"{SYMBOL} — Basis forward (F - Spot) par echeance",
        xaxis_title="DTE (jours)",
        yaxis_title=f"Basis ({CURRENCY})",
        height=420,
    )
    fig.show()

## 3 · Inversion de la volatilité implicite
Chaque quote usable est inversée (Black-76 forward-form) en IV, positionnée en log-moneyness
`k = ln(K/F)`. `outputs.iv_points` porte un `IvPoint` (`iv`, `k`, `total_variance`) par contrat.

In [ ]:
iv_by_expiry = defaultdict(list)
for p in outputs.iv_points:
    instrument = MASTER_BY_KEY[p.contract_key]
    iv_by_expiry[instrument.expiry].append((instrument, p))
print(f"IvPoints solved : {len(outputs.iv_points)}  sur {len(iv_by_expiry)} echeance(s)")

fig = go.Figure()
for i, (expiry, rows) in enumerate(sorted(iv_by_expiry.items())):
    color = DISCRETE[i % len(DISCRETE)]
    dte = (expiry - AS_OF.date()).days
    for side, right, dash in [("Call", "C", "solid"), ("Put", "P", "dot")]:
        pts = sorted(
            [(inst, p) for inst, p in rows if inst.option_right == right],
            key=lambda ip: ip[1].k,
        )
        if not pts:
            continue
        fig.add_trace(
            go.Scatter(
                x=[p.k for _, p in pts],
                y=[p.iv * 100 for _, p in pts],
                mode="lines+markers",
                line=dict(color=color, width=1.8, dash=dash),
                marker=dict(size=4),
                name=f"{expiry} {side}",
                legendgroup=str(expiry),
                legendgrouptitle_text=f"{expiry} ({dte}j)" if side == "Call" else None,
                hovertemplate=(
                    f"<b>{expiry} {side}</b><br>k: %{{x:.3f}}<br>IV: %{{y:.1f}}%<extra></extra>"
                ),
            )
        )
fig.add_vline(x=0, line_dash="dash", line_color=C["slate400"])
fig.update_layout(
    title=f"{SYMBOL} — Smile de volatilite implicite (skew actions)",
    xaxis_title="Log-moneyness ln(K/F)",
    yaxis_title="IV (%)",
    legend=dict(groupclick="toggleitem"),
    height=440,
)
fig.show()

## 4 · Surface SVI
Calibration SVI par échéance (`outputs.surface_parameters` : un jeu de paramètres `a,b,ρ,m,σ`
par maturité). `summarize_surface_parameters` résume chaque slice ; la **surface 3D** se peuple
dès ≥ 2 maturités, en évaluant `SviParams.total_variance(k)` sur une grille `(k, T)`.

In [ ]:
slices = sorted(outputs.surface_parameters, key=lambda s: s.maturity_years)
summary = summarize_surface_parameters(outputs.surface_parameters)
print(f"Slices SVI calibrees : {len(slices)}")
for row in summary:
    dte = int(row.maturity_years * 365)
    rmse = f"RMSE={row.rmse:.5f}" if row.rmse is not None else "RMSE=n/a"
    arb = "arb-free" if row.arb_free else "ARB!"
    print(
        f"  {dte:3d}j  [{row.method.upper():5s}]  n={row.n_points:3d}  "
        f"atm_vol={row.atm_vol:.4f}  {rmse}  [{arb}]"
    )

# Per-slice fit: market total variance (from IV points) vs the calibrated SVI curve.
ncols = min(2, len(slices)) or 1
nrows = (len(slices) + ncols - 1) // ncols
fig = make_subplots(
    rows=nrows,
    cols=ncols,
    horizontal_spacing=0.08,
    vertical_spacing=0.14,
    subplot_titles=[f"{int(s.maturity_years * 365)}j" for s in slices],
)
for idx, sp in enumerate(slices):
    row, col = idx // ncols + 1, idx % ncols + 1
    market = [
        (p.k, p.total_variance)
        for p in outputs.iv_points
        if abs(
            maturity_years(MASTER_BY_KEY[p.contract_key].expiry, AS_OF.date()) - sp.maturity_years
        )
        < 1e-6
    ]
    if market:
        mk = sorted(market)
        fig.add_trace(
            go.Scatter(
                x=[k for k, _ in mk],
                y=[w for _, w in mk],
                mode="markers",
                marker=dict(
                    color=C["blue"], size=7, opacity=0.85, line=dict(color=C["white"], width=0.5)
                ),
                name="Marche" if idx == 0 else None,
                showlegend=(idx == 0),
                hovertemplate="k: %{x:.3f}<br>w: %{y:.5f}<extra></extra>",
            ),
            row=row,
            col=col,
        )
    params = SviParams(a=sp.svi_a, b=sp.svi_b, rho=sp.svi_rho, m=sp.svi_m, sigma=sp.svi_sigma)
    k_grid = np.linspace(-0.5, 0.5, 200)
    fig.add_trace(
        go.Scatter(
            x=k_grid,
            y=[params.total_variance(float(k)) for k in k_grid],
            mode="lines",
            line=dict(color=C["red"], width=2),
            name="SVI fit" if idx == 0 else None,
            showlegend=(idx == 0),
            hovertemplate="k: %{x:.3f}<br>w: %{y:.5f}<extra></extra>",
        ),
        row=row,
        col=col,
    )
fig.update_yaxes(tickformat=".4f")
fig.update_layout(
    title=f"{SYMBOL} — Calibration SVI : variance totale vs log-moneyness",
    height=320 * nrows,
)
fig.show()

In [ ]:
if len(slices) < 2:
    print(f"Surface 3D non tracee : {len(slices)} maturite(s). Fournir un sample multi-echeances.")
else:
    k_axis = np.linspace(-0.4, 0.4, 60)
    t_axis = np.array([s.maturity_years for s in slices])
    iv_grid = np.empty((len(slices), len(k_axis)))
    for i, sp in enumerate(slices):
        params = SviParams(a=sp.svi_a, b=sp.svi_b, rho=sp.svi_rho, m=sp.svi_m, sigma=sp.svi_sigma)
        tv = np.array([params.total_variance(float(k)) for k in k_axis])
        iv_grid[i, :] = np.sqrt(np.maximum(tv, 0.0) / sp.maturity_years)
    kk, tt = np.meshgrid(k_axis, t_axis)
    fig = go.Figure(
        go.Surface(
            x=kk,
            y=tt,
            z=iv_grid,
            colorscale=SURFACE_COLORSCALE,
            opacity=0.9,
            colorbar=dict(
                title=dict(text="IV", side="right"), tickformat=".0%", thickness=16, len=0.7
            ),
            hovertemplate="k: %{x:.3f}<br>T: %{y:.3f}an<br>IV: %{z:.1%}<extra></extra>",
        )
    )
    fig.update_layout(
        title=f"{SYMBOL} — Surface de volatilite implicite (SVI)",
        height=620,
        scene=dict(
            xaxis=dict(title="Log-Moneyness", backgroundcolor=C["slate100"], showbackground=True),
            yaxis=dict(title="Maturite (an)", backgroundcolor=C["slate100"], showbackground=True),
            zaxis=dict(
                title="IV", tickformat=".0%", backgroundcolor=C["slate100"], showbackground=True
            ),
            camera=SURFACE_CAMERA,
            aspectmode="manual",
            aspectratio=SURFACE_ASPECT,
        ),
        margin=dict(l=0, r=0, t=56, b=0),
    )
    fig.show()
    print(
        f"fig.to_json() OK — {len(fig.to_json().encode()) / 1024:.1f} KB "
        "(pret FastAPI -> react-plotly.js)"
    )

## 5 · IBKR ne fournit aucun Greek — le moteur reconstruit tout
Contrairement à Saxo, IBKR ne renvoie **aucun** Greek ni IV sur le flux delayed gratuit. Il n'y
a donc **rien à confronter** : forward, IV et surface sont calculés **uniquement** depuis les
quotes (bid/ask/last) — c'est le mode nominal, identique à Deribit. La reconstruction *est* la
source de vérité.

In [ ]:
from algotrading.infra.storage import events_from_json as _efj

raw = _efj(SAMPLE.read_text(encoding="utf-8"))
broker_greeks = sorted(
    {e.field_name for e in raw if e.field_name in ("delta", "gamma", "vega", "theta", "mark_iv")}
)
print(f"Champs Greeks/IV fournis par IBKR : {broker_greeks or 'AUCUN'}")
print("-> Le moteur reconstruit forward + IV + surface depuis les seules quotes.")

## 6 · (Optionnel) Flux réel via `run_provider_flow` — credential-gated
Le chemin flux-réel, **désactivé par défaut**. Il remplace l'ancien `IbkrFlow` par le seam de
collecte unifié post-merge : `orchestration.ProviderCapture` + `run_provider_flow` capturent le
feed IBKR dans la **même** couche raw, puis `reconstruct_day` calcule la surface — exactement le
chemin ci-dessus, mais alimenté en live. **Nécessite un IB Gateway en marche** (et l'extra
`ibkr`). Laisser `RUN_LIVE = False` pour un notebook qui tourne sans aucun identifiant.

In [ ]:
RUN_LIVE = False  # passer a True uniquement avec un IB Gateway accessible

if not RUN_LIVE:
    print(
        "Live desactive (RUN_LIVE=False) — le notebook tourne "
        "entierement sur l'echantillon committe."
    )
    print("Pour le live : IB Gateway en marche + `uv sync --extra ibkr`, puis RUN_LIVE=True.")
else:
    from algotrading.infra.connectivity import Clock  # noqa: F401
    from algotrading.infra.orchestration import (  # noqa: F401
        ProviderCapture,
        run_provider_flow,
    )
    from algotrading.infra.storage import ParquetStore  # noqa: F401
    from algotrading.infra_ibkr.collectors import (  # noqa: F401
        CpRestDiscovery,
        CpRestMarketDataAdapter,
    )

    # Build the IBKR adapter + the chain to subscribe (discovery), then drive the feed.
    # The adapter/discovery/clock/feed-driver are dependency-injected — see infra_ibkr.collectors
    # and orchestration.provider_flow for the exact wiring against your Gateway.
    raise NotImplementedError(
        "Wire CpRestDiscovery -> subscribe keys, CpRestMarketDataAdapter, a Clock and a feed "
        "driver into ProviderCapture(provider='IBKR', adapter=..., subscribe=..., drive=...), "
        "call run_provider_flow(store=ParquetStore(...), providers=[capture], trade_date=..., "
        "clock=..., correlation_id=...), then reconstruct_day(store, trade_date, ...) as above."
    )

## Récapitulatif de la pile

| Étape | Sortie | Module moteur |
|---|---|---|
| Échantillon committé | events (offline) | `storage.events_from_json` |
| Rejeu → calcul | `ActorOutputs` | `orchestration.reconstruction.reconstruct_day` |
| Snapshot + QC | spot ref, spread, complétude | `snapshots` / `qc` (dans l'acteur) |
| Forward | parité put-call | `forwards` |
| Inversion IV | smile par échéance | `iv` |
| Surface SVI | slices + surface 3D | `surfaces` |
| Greeks broker | IBKR n'en fournit aucun → tout reconstruit | — |
| Flux réel (option) | capture live → même calcul | `orchestration.run_provider_flow` |

Reconstruction **déterministe** depuis une donnée réelle committée — mêmes inputs → mêmes
sorties, sans broker ni réseau. Le notebook n'**importe et appelle** que des moteurs testés.